<a href="https://colab.research.google.com/github/sriharikrishna/JDEV-Tutorial/blob/python/03CtxDbg/03CtxDbg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Context and Debugging with JAX

This section demonstrates how to define functions and compute their derivatives with JAX. JAX supports efficient numerical computation and automatic differentiation.

In [ ]:
import jax
import jax.numpy as jnp

def some_tmp_val_jax(x):
    return x**2 / 2.0 + x + 1.0

# Returns a tuple: (radius, outcos, outsin)
def get_prod_sum_jax(in_val):
    tmp = some_tmp_val_jax(in_val)
    outcos = jnp.cos(tmp)
    outsin = jnp.sin(tmp)

    rad = outcos**2 + outsin**2
    return rad, outcos, outsin


# Example usage:
innum = 2.0
rad, outcos, outsin = get_prod_sum_jax(innum)

print(f"Input: {innum}")
print(f"Radius (outcos^2 + outsin^2) = {rad}")
print(f"outcos = {outcos}")
print(f"outsin = {outsin}")

Input: 2.0
Radius (outcos^2 + outsin^2) = 1.0
outcos = 0.28366219997406006
outsin = -0.9589242935180664


### Forward-Mode Differentiation with `jax.jvp`

[`jax.jvp`](https://jax.readthedocs.io/en/latest/jax.html#jax.jvp) computes $J \cdot v$ for a function $R^n \rightarrow R^m$. It requires the primal input values (the point where derivatives are computed) and a tangent vector $v$ (same shape as the input).

For a scalar input, set the tangent to `1.0` to get standard first derivatives. When the function returns multiple outputs, `jax.jvp` returns both the primal outputs and the corresponding tangent (derivative) for each output.

In [ ]:
# Primal input and tangent (1.0 for d/d_innum)
primal_in = jnp.array(innum)
tangent_in = jnp.array(1.0)

(rad_primal, outcos_primal, outsin_primal), \
(rad_tangent, outcos_tangent, outsin_tangent) = jax.jvp(
    get_prod_sum_jax, (primal_in,), (tangent_in,)
)

print(f"Input: {innum}\n")

print("--- Primal (Function) Values ---")
print(f"Radius (outcos^2 + outsin^2) = {rad_primal}")
print(f"outcos = {outcos_primal}")
print(f"outsin = {outsin_primal}\n")

print("--- Tangent (Derivative) Values (d/d_innum) ---")
print(f"d(Radius)/d_innum = {rad_tangent}")
print(f"d(outcos)/d_innum = {outcos_tangent}")
print(f"d(outsin)/d_innum = {outsin_tangent}")

Input: 2.0

--- Primal (Function) Values ---
Radius (outcos^2 + outsin^2) = 1.0
outcos = 0.28366219997406006
outsin = -0.9589242935180664

--- Tangent (Derivative) Values (d/d_innum) ---
d(Radius)/d_innum = 0.0
d(outcos)/d_innum = 2.876772880554199
d(outsin)/d_innum = 0.8509865999221802


### Checking forward-mode derivatives with `check_grads`

[`jax.test_util.check_grads`](https://docs.jax.dev/en/latest/_autosummary/jax.test_util.check_grads.html) compares automatic differentiation against finite differences. Set `modes=('fwd',)` to check forward-mode (`jax.jvp`) derivatives. If the check passes, the `jax.jvp` tangents above agree with finite-difference estimates.

In [ ]:
from jax.test_util import check_grads

# Verify jax.jvp derivatives against finite differences (forward mode only)
check_grads(get_prod_sum_jax, (jnp.array(innum),), order=1, modes=("fwd",))

print("check_grads passed: jax.jvp results match finite differences.")